# Lambda selection on validation (wave A)

This notebook chooses the consistency weight lambda for the proposed system. It reads only the
`validation/` folder and `run.json` of each wave A job, fetched from the Modal volume with
`scripts/modal/fetch_wave.py --part validation`. It never opens a `dev/` folder, and it records
whether any had been fetched when it ran.

The rule was fixed in issue #25 before the runs: the chosen lambda is the grid value with the
highest mean validation weighted F1 (single pass, kept epoch) over the three seeds, compared at
full precision; only an exactly equal mean goes to the smaller lambda. Lambda 0 is the comparison
system and is not in the grid.

It writes `lambda_selection.json`, `tables/validation_per_seed.csv` and
`tables/validation_grid.csv` into the run folder. Commit and push them before fetching dev.

In [ ]:
TRACK = 1
RUN_FOLDER = "2026-11-25-decoder-final-track1"
WAVE_A_RUN_ID = "2026-11-25-decoder-final-wave-a"
WAVE_A_APP_ID = "ap-eOvzhee5YGcEo56Mn1oenz"
GRID = (0.25, 0.5, 1.0, 2.0)
SEEDS = (13, 42, 1234)
COMMAND = "PYTHONIOENCODING=utf-8 uv run jupyter execute notebooks/assignment-3/01_lambda_selection.ipynb"

In [ ]:
import json
import statistics
from datetime import datetime, timezone

import pandas as pd

from dico_nli.data import LABELS, REPO_ROOT
from dico_nli.experiments import EXPERIMENTS_ROOT, data_commit
from dico_nli.method.reversal import symmetrize
from dico_nli.scoring import score, write_predictions

run_dir = EXPERIMENTS_ROOT / RUN_FOLDER
tables_dir = run_dir / "tables"
tables_dir.mkdir(exist_ok=True)
wave_dir = REPO_ROOT / "artifacts" / "decoder-final-track1" / WAVE_A_RUN_ID
scratch = REPO_ROOT / "artifacts" / "decoder-final-track1" / "validation-scores"
jobs = json.loads((run_dir / "wave_a_jobs.json").read_text(encoding="utf-8"))

## The runs

One record per job, from the copy of `run.json` inside its `validation/` folder. Every job must
come from the same source code and a clean tree.

In [ ]:
records = {}
for folder in sorted(path for path in wave_dir.iterdir() if path.is_dir()):
    records[folder.name] = json.loads((folder / "validation" / "run.json").read_text(encoding="utf-8"))

assert len(records) == len(jobs) == 16, (len(records), len(jobs))
source_digests = {record["code"]["source_sha256"] for record in records.values()}
repo_commits = {record["code"]["repo_commit"] for record in records.values()}
assert len(source_digests) == 1 and len(repo_commits) == 1, (source_digests, repo_commits)
assert all(record["code"]["repo_dirty"] is False for record in records.values())
dev_folders_present = sorted(name for name in records if (wave_dir / name / "dev").exists())
print(f"{len(records)} jobs, source {next(iter(source_digests))[:12]}, commit {next(iter(repo_commits))[:12]}")
print("dev folders fetched locally:", dev_folders_present or "none")

## Validation scores

Each job saved the validation probabilities of its kept epoch in both orders. The single-pass
prediction is the most likely label of the given order; the symmetrized one averages the two
orders (`dico_nli.method.reversal.symmetrize`). Both go through the official scorer against the
job's own validation reference. The single-pass weighted F1 must equal the value the training
loop logged at the kept epoch.

In [ ]:
columns = list(LABELS)
rows = []
for name, record in records.items():
    validation = wave_dir / name / "validation"
    forward = pd.read_csv(validation / "probabilities_forward.csv")
    reverse = pd.read_csv(validation / "probabilities_reverse.csv")
    assert list(forward["instance_id"]) == list(reverse["instance_id"])
    assert len(forward) == record["holdout_size"]
    decoded = {
        "single": forward[columns].to_numpy().argmax(axis=1),
        "symmetrized": symmetrize(forward[columns].to_numpy(), reverse[columns].to_numpy()).argmax(axis=1),
    }
    for decoding, best in decoded.items():
        out = scratch / f"{name}__{decoding}"
        predictions = write_predictions(out / "predictions.csv", {i: LABELS[k] for i, k in zip(forward["instance_id"], best)})
        scores = score(validation / "reference.csv", predictions, out)
        rows.append({
            "system": name,
            "mode": record["mode"],
            "lambda": record["config"]["consistency_weight"] if record["mode"] == "lora" else None,
            "seed": record["split_seed"],
            "kept_epoch": record["best_epoch"],
            "validation_size": record["holdout_size"],
            "decoding": decoding,
            "weighted_f1": scores.weighted_f1,
            "soft_cons": scores.soft_cons,
            "hard_cons": scores.hard_cons,
        })
        if decoding == "single" and record["mode"] == "lora":
            logged = record["history"][record["best_epoch"] - 1]["holdout_weighted_f1"]
            assert abs(scores.weighted_f1 - logged) < 1e-9, (name, scores.weighted_f1, logged)

per_seed = pd.DataFrame(rows)
per_seed.to_csv(tables_dir / "validation_per_seed.csv", index=False)
per_seed

## The grid

Mean and sample standard deviation over the three seeds, for every lambda and both decodings.
Lambda 0 and the zero-shot model (one deterministic run, scored on seed 13's validation split)
are shown for reference and take no part in the choice.

In [ ]:
grid_rows = []
trained = per_seed[per_seed["mode"] == "lora"]
for (weight, decoding), group in trained.groupby(["lambda", "decoding"]):
    group = group.set_index("seed").loc[list(SEEDS)]
    row = {"system": f"lora lambda {weight:g}", "lambda": weight, "decoding": decoding, "seeds": len(group)}
    for metric in ("weighted_f1", "soft_cons", "hard_cons"):
        row[f"{metric}_mean"] = statistics.fmean(group[metric])
        row[f"{metric}_sd"] = statistics.stdev(group[metric])
    row.update({f"kept_epoch_seed{seed}": int(group.loc[seed, "kept_epoch"]) for seed in SEEDS})
    grid_rows.append(row)
for _, zero_shot in per_seed[per_seed["mode"] == "zero-shot"].iterrows():
    row = {"system": "zero-shot", "lambda": None, "decoding": zero_shot["decoding"], "seeds": 1}
    row.update({f"{metric}_mean": zero_shot[metric] for metric in ("weighted_f1", "soft_cons", "hard_cons")})
    grid_rows.append(row)

grid = pd.DataFrame(grid_rows)
grid.to_csv(tables_dir / "validation_grid.csv", index=False)
grid.round(4)

## The choice

The rule from issue #25, applied to the single-pass means at full precision.

In [ ]:
single = grid[(grid["decoding"] == "single") & grid["lambda"].isin(GRID)]
assert sorted(single["lambda"]) == sorted(GRID)
means = {float(row["lambda"]): float(row["weighted_f1_mean"]) for _, row in single.iterrows()}
best = max(means.values())
chosen = min(weight for weight, mean in means.items() if mean == best)  # an exact tie goes to the smaller lambda

selection = {
    "chosen_lambda": chosen,
    "rule": "highest mean validation weighted F1 (single pass, kept epoch) over seeds 13, 42 and 1234, "
            "compared at full precision; an exactly equal mean goes to the smaller lambda; lambda 0 is not in the grid",
    "grid": list(GRID),
    "seeds": list(SEEDS),
    "mean_validation_weighted_f1_single": {f"{weight:g}": mean for weight, mean in sorted(means.items())},
    "per_seed_validation_weighted_f1_single": {
        f"{weight:g}": {str(seed): float(trained[(trained["lambda"] == weight) & (trained["seed"] == seed)
                                                  & (trained["decoding"] == "single")]["weighted_f1"].iloc[0])
                        for seed in SEEDS}
        for weight in (0.0, *GRID)
    },
    "kept_epoch": {
        f"{weight:g}": {str(seed): int(trained[(trained["lambda"] == weight) & (trained["seed"] == seed)]["kept_epoch"].iloc[0])
                        for seed in SEEDS}
        for weight in (0.0, *GRID)
    },
    "chosen_is_grid_edge": chosen in (min(GRID), max(GRID)),
    "wave_a_run_id": WAVE_A_RUN_ID,
    "wave_a_app_id": WAVE_A_APP_ID,
    "wave_a_repo_commit": next(iter(repo_commits)),
    "source_sha256": next(iter(source_digests)),
    "data_commit": data_commit(),
    "track": TRACK,
    "inputs": "validation/ and run.json of each wave A job only",
    "dev_folders_fetched_locally_when_run": dev_folders_present,
    "run_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "command": COMMAND,
}
(run_dir / "lambda_selection.json").write_text(json.dumps(selection, indent=2) + "\n", encoding="utf-8")
print(json.dumps(selection, indent=2))